[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1dBzOTW9vE4bnNsw4tYzOBKNPSFkbRyrg#scrollTo=5fd6274f)


# Molecular Weight Calculator
**Purpose**: Google Colab is a free, cloud-based Jupyter notebook environment that lets you write and run Python code directly in your browser — no installation needed! This Notebook is designed to calculate molecule weights for chemical compounds. This is a starting point to a stoichiometry notebook, which will perform grams to moles calculations, solution stoichiometry, and limited reagent problems.
**Libraries**

## Resources

| Library                 | Purpose                                           | Citation                                                                                                                                                                                                            |
| :---------------------- | :------------------------------------------------ | :---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **mendeleev**           | Provides access to detailed<br>periodic table data.  | [mendeleev library on PyPI](https://pypi.org/project/mendeleev/)                                                                                                                                                        |
| **re**  | Enables regular expression operations<br>for pattern matching and<br>string manipulation. | [Python re module documentation](https://docs.python.org/3/library/re.html)                                                                                                                                           |
| **Python Standard Library** | Core functionalities for<br>general-purpose programming. | [Python Documentation](https://docs.python.org/3/). |

**Status with Date:**
First attempt in CHMG 131 9.18.26

**License**

<img src="https://github.com/codeBMB/April26_Rutgers-RCSB/raw/main/images/by-nc-sa.png" width="100" alttext="[CC BY-NC-SA](https://creativecommons.org/licenses/by-nc-sa/4.0/)"/>
 This license enables reusers to distribute, remix, adapt, and build upon the material in any medium or format for noncommercial purposes only, and only so long as attribution is given to the creator. If you remix, adapt, or build upon the material, you must license the modified material under identical terms. CC BY-NC-SA includes the following elements:
 BY: credit must be given to the creator.
 NC: Only noncommercial uses of the work are permitted.
 SA: Adaptations must be shared under the same terms.

## Steps
1.   Enter chemical formulas with the proper syntax.
2.   Generate results for as many chemical compounds as you need.

## Questions
*   How can I enter chemical formulas properly without including subscripts?
*   Can I go further in solving stoichiometry problems in Python?

## Learning Objectives
*   Learn to write chemical formulas that Python can interpret
*   Install packages that enable Python to do the calculations
*   Work with a Python dictionary for the elements of the periodic table
*   Use a function to calculate molecular weights

## Section 1 — Making a Copy

When you open someone else's notebook, it's **read-only**. Making a copy saves it to your own Google Drive so you can edit and save changes.

**How to make a copy:**
1. Click **File → Save a copy in Drive** from the top menu.
2. A new tab will open with *your own editable copy* — it's automatically named **"Copy of [original name]"**.
3. You can rename it by clicking on the title at the top of the page.

> **Always** make a copy before editing a shared notebook! Go ahead and make your copy now before continuing.

## Section 2 — Saving Your Work

Colab **auto-saves** your notebook to Google Drive every few minutes, but you can also save manually.

**Saving options:**
- **Auto-save:** Happens automatically in the background.
- **Manual save:** Press `Ctrl + S` (Windows/Linux) or `Cmd + S` (Mac), or go to **File → Save**.
- **Save a copy in Drive:** File → Save a copy in Drive (creates a new file).
- **Download:** File → Download → Download .ipynb (saves a local copy to your computer).

> All saved notebooks live in your Google Drive under **Colab Notebooks** unless you move them.

In [ ]:
# This code will install the required libraries.
# The %%capture command hides the details of the installation process.
%%capture
!pip install mendeleev

## The Element Dictionary

This dictionary enables the code to extract information about each of the elements of the periodic table. For this notebook, we will only be using the atomic weights, but we may use other features in future notebooks.

In [ ]:
# @title This cell contains the code for a dictionary of the elements of the periodic table.

from mendeleev import element

# Data dictionary for all 118 elements

periodic_table = {}       # this command creates an empty dictionary

# This command fetches all elements from the mandeleev package
elements = element(list(range(1, 119)))

# This for loop creates the dictionary with 12 different descriptors for each element

for el in elements:
    periodic_table[el.symbol] = {
        "name": el.name,
        "atomic_number": el.atomic_number,
        "atomic_weight": el.atomic_weight,
        "group": el.group_id,
        "period": el.period,
        "block": el.block,
        "category": el.series,
        "electron_configuration": el.econf,
        "electronegativity": el.electronegativity(),
        "density": el.density,
        "discovered_by": el.discoverers,
        "description": el.description,

        # These two throw a warning "Xy has multiple allotropes" for P, S, Se, and Sn
        # "melting_point": el.melting_point,
        # "boiling_point": el.boiling_point,
    }

# Test: Silicon, 14, 14. You can change this and test another element.
# You could also ask it to provide any other descriptor in the dictionary.

id = periodic_table["W"]
print(id["name"])
print(id["atomic_number"])
print(id["group"])
print(id["electron_configuration"])

### Molecular Weight Calculation

To calculate the molecular weight of a compound, we need to:

1.  **Parse the chemical formula**: Identify each element symbol and its corresponding count in the formula.
2.  **Retrieve atomic weights**: For each element, look up its atomic weight in the `periodic_table` dictionary.
3.  **Sum the weights**: Multiply each element's atomic weight by its count and sum these values to get the total molecular weight. Special consideration is needed for parenthetical groups (e.g., in `(OH)2`).

In [ ]:
# @title This cell contains the code for the function, calculate_molecular_weight

import re

def calculate_molecular_weight(formula):
    """
    Calculates the molecular weight of a chemical compound from its formula with input validation,
    including support for hydrates (e.g., 'MgSO4*7H2O').

    Args:
        formula (str): The chemical formula of the compound (e.g., 'H2O', 'MgSO4', 'Ca(OH)2', 'MgSO4*7H2O').

    Returns:
        float: The molecular weight of the compound.

    Raises:
        ValueError: If the formula contains invalid symbols, syntax, or unknown elements.
    """

    # Nested helper function to parse segments (elements, counts, parentheses)
    def parse_segment(segment_to_parse):
        sub_weight = 0.0
        i = 0
        while i < len(segment_to_parse):
            # Handle parentheses recursively
            if segment_to_parse[i] == '(':
                parenting_level = 1
                j = i + 1
                while j < len(segment_to_parse) and parenting_level > 0:
                    if segment_to_parse[j] == '(': parenting_level += 1
                    elif segment_to_parse[j] == ')': parenting_level -= 1
                    j += 1

                if parenting_level != 0: # Unmatched parentheses
                    raise ValueError(f"Unmatched parentheses in segment: {segment_to_parse[i:]}")

                group_content = segment_to_parse[i+1:j-1]
                i = j # Move past the group and its closing parenthesis

                count_str = ''
                while i < len(segment_to_parse) and segment_to_parse[i].isdigit():
                    count_str += segment_to_parse[i]
                    i += 1
                group_count = int(count_str) if count_str else 1

                sub_weight += parse_segment(group_content) * group_count

                if i < len(segment_to_parse) and re.match(r'[A-Z][a-z]?', segment_to_parse[i:]):
                    element_str = re.match(r'[A-Z][a-z]?', segment_to_parse[i:]).group(0)
                    raise ValueError(f"Invalid syntax: Element '{element_str}' cannot immediately follow a parenthetical group with a count in formula: '{formula}'")
                continue

            # Handle elements and their counts
            element_match = re.match(r'([A-Z][a-z]?)([0-9]*)', segment_to_parse[i:])
            if element_match:
                symbol = element_match.group(1)
                count_str = element_match.group(2)
                count = int(count_str) if count_str else 1

                if symbol in periodic_table:
                    if periodic_table[symbol]["atomic_weight"] is None:
                        raise ValueError(f"Atomic weight for element '{symbol}' is not available.")
                    sub_weight += periodic_table[symbol]["atomic_weight"] * count
                else:
                    raise ValueError(f"Unknown element symbol: '{symbol}' found in formula: {formula}")

                i += len(element_match.group(0))
            else:
                raise ValueError(f"Invalid sequence or syntax near '{segment_to_parse[i:]}' in formula: {formula}")
        return sub_weight

    # --- Main logic for calculate_molecular_weight ---
    total_molecular_weight = 0.0
    main_compound_formula = formula
    hydrate_component_weight = 0.0

    # Check for hydrate part (e.g., *7H2O)
    if '*' in formula:
        parts = formula.split('*', 1) # Split only on the first '*'
        main_compound_formula = parts[0]
        hydrate_part_str = parts[1].strip()

        hydrate_match = re.match(r'([0-9]*)(H2O)', hydrate_part_str)
        if not hydrate_match:
            raise ValueError(f"Invalid hydrate format '{hydrate_part_str}' in formula: {formula}. Expected N H2O or H2O.")

        coeff_str = hydrate_match.group(1)

        hydrate_coefficient = int(coeff_str) if coeff_str else 1 # Default to 1 for H2O

        # Calculate molecular weight of H2O using the nested parse_segment
        try:
            water_molecular_weight = parse_segment('H2O')
        except ValueError as e:
            raise ValueError(f"Internal error calculating molecular weight of H2O for hydrate: {e}")

        hydrate_component_weight = hydrate_coefficient * water_molecular_weight

    # Validate main compound formula (excluding the hydrate part if present)
    if not re.fullmatch(r'[A-Za-z0-9()]+', main_compound_formula):
        raise ValueError(f"Invalid characters found in main compound formula: {main_compound_formula}")
    elif not main_compound_formula: # Ensure main compound is not empty if hydrate is present
        raise ValueError(f"Main compound formula is empty in: {formula}")


    # Calculate molecular weight of the main compound
    total_molecular_weight += parse_segment(main_compound_formula)
    total_molecular_weight += hydrate_component_weight

    return total_molecular_weight

## Calculating Molecular Weights

You can use the next cell to calculate the molecular weight of any single chemical compound.
1. You just need to enter the chemical formula for your compound in the place where H2SO4 is shown.
1. You need to enter the formula as simple text with no additional formatting, like subscripts or superscripts. Here is an example. In a report, I would write sulfuric acid as $H_2SO_4$. For this notebook, simply write the chemical name as a simple string. Remember, it is a string and needs to be in quotes.
1. Here are some examples of chemical formulas you might want to use, with the way they appear in textbooks and the way we will enter them.
1. In the three coding cells below the table, you have the option of (a) assigning the name of your compound to a variable and using that variable to calculate the molecular weight; (b) calculating the molecular weight of the compound directly; or (c) using a for loop to calculate a the molecular weights for a list of chemical compounds.
**Note that the chemical compounds are always entered in quotes, as strings.**

### Chemical Formula Examples

| Chemical Name                 | Chemical Formula (LaTeX)     | Chemical Formula (Text) |
| :---------------------------- | :--------------------------- | :---------------------- |
| Sulfuric Acid                 | $H_2SO_4$                    | H2SO4                   |
| Magnesium Sulfate Heptahydrate| $MgSO_4 \cdot 7H_2O$         | MgSO4*7H2O              |
| Iron(III) oxide               | $Fe_2O_3$                    | Fe2O3                   |
| Iron(III) Sulfate               | $Fe_2(SO_4)_3$                    | Fe2(SO4)3                   |
| Urea              |  $CO(NH_2)_2$ | CO(NH2)2                   |

### Error Checking and Validation in Molecular Weight Calculation

To ensure accurate molecular weight calculations and robust handling of various chemical formula inputs, the `calculate_molecular_weight` function now includes several layers of error checking and validation:

1.  **Invalid Characters**: The function first checks if the input formula contains any characters that are not alphanumeric or parentheses. For example, `H@O` will immediately raise an error.
2.  **Unknown Element Symbols**: If an element symbol (e.g., 'X' in `H2X`) is encountered that is not present in our `periodic_table` dictionary, a `ValueError` is raised, clearly indicating the unknown element.
3.  **Missing Atomic Weights**: Although unlikely with the `mendeleev` library, if an element exists in the periodic table but its atomic weight is `None`, an error is raised.
4.  **Unmatched Parentheses**: The parser carefully tracks the opening and closing of parentheses. If an opening parenthesis `(` does not have a corresponding closing parenthesis `)` (e.g., `Ca(OH2`), a `ValueError` for unmatched parentheses is raised.
5.  **Invalid Syntax (Element after Parenthetical Group)**: A crucial validation rule was added to catch syntactically incorrect but potentially parsable formulas like `(OH)2H`. In proper chemical notation, an element cannot directly follow a parenthetical group that has a subscript count. The function now explicitly checks for this and raises an error to prevent misinterpretation of such formulas.
6.  **Invalid Sequence/Syntax**: If any part of the formula does not conform to the expected pattern of element symbols followed by optional counts, or valid parenthetical groups, a general `ValueError` for invalid sequence or syntax is raised. This catches cases like an empty string `""` or just a digit `"2"` as a formula, or lowercase element symbols like `mgso4`.

These checks help guide the user to provide correctly formatted chemical formulas, making the tool more reliable for chemical calculations.

In [ ]:
# Example usage with error handling:
formula1 = "H2O"
formula2 = "MgSO4"
formula3 = "Ca(OH)2"
formula4 = "C6H12O6"
formula_hydrate = "MgSO4*7H2O"
formula_monohydrate = "CuSO4*H2O"
formula_invalid_symbol = "H2X"
formula_invalid_syntax = "(OH)2H"
formula_unmatched_paren = "Ca(OH2"

# Valid formulas
formulas_to_test = {
    "H2O": "H2O",
    "MgSO4": "MgSO4",
    "Ca(OH)2": "Ca(OH)2",
    "C6H12O6": "C6H12O6",
    "Element with no count after parenthesis": "Ca(OH)", # This is valid, implies Ca(OH)1
    "Magnesium Sulfate Heptahydrate": "MgSO4*7H2O",
    "Copper Sulfate Monohydrate": "CuSO4*H2O"
}

for name, formula in formulas_to_test.items():
    try:
        mw = calculate_molecular_weight(formula)
        print(f"Molecular weight of {name} ({formula}): {mw:.3f} g/mol")
    except ValueError as e:
        print(f"Error calculating molecular weight for {name} ({formula}): {e}")

print("\n--- Testing invalid formulas ---")

# Invalid formulas
formulas_to_test_invalid = {
    "Invalid Symbol": "H2X",
    "Invalid Syntax (Element after paren count)": "(OH)2H", # Now raises an error
    "Unmatched Parentheses": "Ca(OH2",
    "Invalid Characters": "H@O",
    "Empty Formula": "",
    "Just a digit": "2",
    "Lowercase element symbol": "mgso4",
    "Invalid Hydrate Format": "MgSO4*7H3O" # Invalid hydrate example
}

for name, formula in formulas_to_test_invalid.items():
    try:
        mw = calculate_molecular_weight(formula)
        print(f"Molecular weight of {name} ({formula}): {mw:.3f} g/mol")
    except ValueError as e:
        print(f"Error calculating molecular weight for {name} ({formula}): {e}")

In [ ]:
# To calculate a molecular weight, use this approach
compound = "Fe2(SO4)3"
mw = calculate_molecular_weight(compound)
print(f"Molecular weight of {compound} : {mw:.3f} g/mol")

In [ ]:
# Or you could just enter the chemical formula directly
mw = calculate_molecular_weight("H2SO4")
print(f"Molecular weight of H2SO4 : {mw:.3f} g/mol")

In [ ]:
# Or you could generate a list of chemicals and get their molecular weights
chemical_list = ["H2SO4", "MgSO4", "Ca(OH)2", "C6H12O6", "KMnO4", "NaCH3COO"]
for chemical in chemical_list:
    mw = calculate_molecular_weight(chemical)
    print(f"Molecular weight of {chemical} : {mw:.3f} g/mol")